# 02 - PDF text extraction
Goal: extract clean body_text from PDFs; fall back to abstract for failures

Input: data/raw/raw_papers.jsonl + data/raw/pdfs/
Output: data/processed/extracted_papers.jsonl

In [ ]:
!pip install -q PyMuPDF jsonlines

from google.colab import drive
drive.mount('/content/drive')

import fitz, json, re, os
from pathlib import Path
from tqdm import tqdm

ROOT      = '/content/drive/MyDrive/arxiv-llm-project'
RAW_JSONL = f'{ROOT}/data/raw/raw_papers.jsonl'
PDF_DIR   = Path(f'{ROOT}/data/raw/pdfs')
OUT_JSONL = f'{ROOT}/data/processed/extracted_papers.jsonl'
os.makedirs(f'{ROOT}/data/processed', exist_ok=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 39.8 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
REF_PAT   = re.compile(r'\n(References|Bibliography)\n', re.I)
FIG_PAT   = re.compile(r'(Figure|Fig\.)\s+\d+[.:][^\n]*', re.I)
MATH_PAT  = re.compile(r'(\$\$.*?\$\$|\\begin\{.*?\}.*?\\end\{.*?\})', re.S)
GARBLE    = re.compile(r'[^\x00-\x7F]{5,}')

def is_garbled(text):
    hits = len(GARBLE.findall(text))
    return (hits / max(1, len(text) / 100)) > 2

def extract(pdf_path):
    try:
        doc  = fitz.open(pdf_path)
        raw  = "\n".join(p.get_text("text") for p in doc)
        doc.close()
        if len(raw) < 500 or is_garbled(raw):
            return None, "abstract_only"
        m = REF_PAT.search(raw)
        if m: raw = raw[:m.start()]
        raw = FIG_PAT.sub("", raw)
        raw = MATH_PAT.sub("", raw)
        clean = re.sub(r'\n{3,}', '\n\n', raw).strip()
        return clean, "full_text"
    except Exception as e:
        print(f"  ✗ {pdf_path.stem}: {e}")
        return None, "abstract_only"

In [ ]:
papers = [json.loads(l) for l in open(RAW_JSONL)]
full_count = 0

with open(OUT_JSONL, "w") as out:
    for p in tqdm(papers, desc="Extracting"):
        pdf = PDF_DIR / f"{p['paper_id']}.pdf"
        body, src = extract(pdf) if pdf.exists() else (None, "abstract_only")
        p["body_text"] = body
        p["source"]    = src
        if src == "full_text": full_count += 1
        out.write(json.dumps(p) + "\n")

print(f"Full-text: {full_count}/{len(papers)}")
print(f"Abstract-only: {len(papers)-full_count}/{len(papers)}")

Extracting: 100%|██████████| 25000/25000 [04:04<00:00, 102.12it/s] 

Full-text: 50/25000
Abstract-only: 24950/25000


In [ ]:
import random
extracted = [json.loads(l) for l in open(OUT_JSONL)]
full_text = [p for p in extracted if p["source"] == "full_text"]
sample = random.sample(full_text, min(3, len(full_text)))
for p in sample:
  print(f"--- {p['paper_id']} ---")
  print(f"Title: {p['title'][:80]}")
  print(f"Body length: {len(p['body_text'])} chars")
  print(p["body_text"][:300])
  print()
  lengths = [len(p["body_text"]) for p in full_text]
  print(f"Avg body length: {sum(lengths)//len(lengths)} chars")
  print(f"Min: {min(lengths)} Max: {max(lengths)}")

--- 2201.00377v1 ---
Title: Parkour Spot ID: Feature Matching in Satellite and Street view images using Deep
Body length: 17750 chars
Parkour Spot ID: Feature Matching in Satellite and
Street view images using Deep Learning
Jo˜ao Morais, Kaushal Rathi, Bhuvaneshwar Mohan and Shantanu Rajesh
joao, kaushalr, bhuvanm, shantanurajesh@asu.edu
Abstract—How to ﬁnd places that are not indexed by Google
Maps? We propose an intuitive method

Avg body length: 57830 chars
Min: 14898 Max: 678146
--- 2201.00418v2 ---
Title: Succinct Differentiation of Disparate Boosting Ensemble Learning Methods for Pro
Body length: 21785 chars
arXiv:2201.00418v2  [cs.LG]  14 Aug 2022
SUCCINCT DIFFERENTIATION OF DISPARATE BOOSTING
ENSEMBLE LEARNING METHODS FOR PROGNOSTICATION OF
POLYCYSTIC OVARY SYNDROME DIAGNOSIS
MODIFIED VERSION OF PUBLISHED ARTICLE
Abhishek Gupta
Department of EXTC
University of Mumbai
Mumbai, 400032, India
abhishekgupt

Avg body length: 57830 chars
Min: 14898 Max: 678146
--- 2201.00230v1 ---
Tit